# E8 bootstrap sampler on cosets of `E8/2E8`

I follow the construction style already used in `sampling_E8.sage`, but extends it to:

1. sample from shifted cosets `E8 + t`,
2. bootstrap sampling from `r + 2E8` using `2 * D_{E8 + r/2, s/2}`
3. test using explicit representatives of `E8/2E8`.

Model used here: `E8 = C + 2*Z^8`, where `C = RM(1,3)`.


In [ ]:
from sage.stats.distributions.discrete_gaussian_integer import DiscreteGaussianDistributionIntegerSampler as DGaussZ
from sage.all import vector, matrix, ZZ, QQ, codes
from mpmath import mp, exp, jtheta, sqrt, pi, log

import itertools
import random
from collections import Counter

mp.dps = 80

In [ ]:
C = codes.BinaryReedMullerCode(1, 3)
CODEWORD_TUPLES = [tuple(int(x) for x in cw) for cw in C]
CODEWORD_SET = set(CODEWORD_TUPLES)

# Construction A basis for E8 = C + 2 Z^8
I2 = [vector(ZZ, [2 if i == j else 0 for j in range(8)]) for i in range(8)]
G = C.generator_matrix()
Grows = [vector(ZZ, [int(x) for x in G.row(i)]) for i in range(G.nrows())]
E8_BASIS = matrix(ZZ, I2 + Grows).row_module().basis_matrix()

print("number of codewords:", len(CODEWORD_TUPLES))
print("E8 basis shape:", E8_BASIS.nrows(), "x", E8_BASIS.ncols())
print(E8_BASIS)


number of codewords: 16
E8 basis shape: 8 x 8
[1 0 0 1 0 1 1 0]
[0 1 0 1 0 1 0 1]
[0 0 1 1 0 0 1 1]
[0 0 0 2 0 0 0 0]
[0 0 0 0 1 1 1 1]
[0 0 0 0 0 2 0 0]
[0 0 0 0 0 0 2 0]
[0 0 0 0 0 0 0 2]


In [3]:
def nome(s):
    return exp(-pi / (s**2))


def rho_2Z_shift(alpha, s):
    # rho_s(2Z + alpha) via Jacobi theta
    q = nome(s)
    z = -2j * pi * alpha / (s**2)
    value = (q ** (alpha**2)) * jtheta(3, z, q**4)
    return float(value.real)


def theta_shifted_E8(s, shift):
    # Theta mass of E8 + shift at width s
    shift_f = [float(a) for a in shift]
    total = 0.0
    for c in CODEWORD_TUPLES:
        prod = 1.0
        for i in range(8):
            prod *= rho_2Z_shift(c[i] + shift_f[i], s)
        total += prod
    return total


def codeword_weights_shifted_E8(s, shift):
    # Relative weights of codeword cosets (2Z^8 + c + shift)
    shift_f = [float(a) for a in shift]
    log_weights = []
    for c in CODEWORD_TUPLES:
        lw = 0.0
        for i in range(8):
            lw += float(log(rho_2Z_shift(c[i] + shift_f[i], s)))
        log_weights.append(lw)
    m = max(log_weights)
    return [float(exp(w - m)) for w in log_weights]


def sample_coord_2Z_coset(alpha, s):
    # Sample one coordinate from D_{2Z + alpha, s}
    sigma = float(s) / (2 * float(sqrt(2 * pi)))
    center = -QQ(alpha) / 2
    z = DGaussZ(sigma=sigma, c=center)()
    return ZZ(2 * z) + QQ(alpha)


def sample_shifted_E8(s, shift):
    # Sample from D_{E8 + shift, s} and return sample + chosen codeword
    if len(shift) != 8:
        raise ValueError("shift must have length 8")

    weights = codeword_weights_shifted_E8(s, shift)
    idx = random.choices(range(len(CODEWORD_TUPLES)), weights=weights, k=1)[0]
    c = CODEWORD_TUPLES[idx]
    x = [sample_coord_2Z_coset(c[i] + shift[i], s) for i in range(8)]
    return vector(QQ, x), c


In [4]:
def e8_mod_2e8_representatives():
    # Representatives of E8/2E8 from the chosen basis: sum b_i * B_i, b_i in {0,1}
    reps = []
    for bits in itertools.product([0, 1], repeat=8):
        r = vector(ZZ, [0] * 8)
        for i, b in enumerate(bits):
            if b:
                r += E8_BASIS.row(i)
        reps.append(r)
    return reps


def in_E8(v):
    if len(v) != 8:
        return False
    for x in v:
        if x not in ZZ:
            return False
    parity = tuple(int(ZZ(x) % 2) for x in v)
    return parity in CODEWORD_SET


def in_2E8(v):
    if len(v) != 8:
        return False
    if any(ZZ(x) % 2 != 0 for x in v):
        return False
    half = vector(ZZ, [ZZ(x) // 2 for x in v])
    return in_E8(half)


def in_rep_plus_2E8(v, rep):
    diff = vector(ZZ, [ZZ(v[i] - rep[i]) for i in range(8)])
    return in_2E8(diff)


def sample_coset_rep_plus_2E8(s, rep):
    # Bootstrap identity: D_{rep + 2E8, s} = 2 * D_{E8 + rep/2, s/2}
    shift = [QQ(rep[i]) / 2 for i in range(8)]
    y, _ = sample_shifted_E8(s / 2, shift)
    return vector(QQ, [2 * yi for yi in y])


def prepare_bootstrap_distribution(s, reps):
    masses = [theta_shifted_E8(s / 2, [QQ(r[i]) / 2 for i in range(8)]) for r in reps]
    total = sum(masses)
    probs = [m / total for m in masses]
    return masses, probs


def sample_E8_via_bootstrap(s, reps, masses=None):
    # Sample from D_{E8, s} by coset selection in E8/2E8, then inner coset sampling
    if masses is None:
        masses, _ = prepare_bootstrap_distribution(s, reps)
    idx = random.choices(range(len(reps)), weights=masses, k=1)[0]
    x = sample_coset_rep_plus_2E8(s, reps[idx])
    return x, idx


In [ ]:
reps = e8_mod_2e8_representatives()
print("number of representatives:", len(reps))
print("all representatives lie in E8:", all(in_E8(r) for r in reps))
print("first 5 representatives:")
for r in reps[:5]:
    print(r)


number of representatives: 256
all representatives lie in E8: True
first 5 representatives:
(0, 0, 0, 0, 0, 0, 0, 0)
(0, 0, 0, 0, 0, 0, 0, 2)
(0, 0, 0, 0, 0, 0, 2, 0)
(0, 0, 0, 0, 0, 0, 2, 2)
(0, 0, 0, 0, 0, 2, 0, 0)


In [6]:
# Test 1: fixed-coset sampler checks
s = 2.2
test_indices = [0, 1, 7, 42, 255]
trials_per_rep = 40

for idx in test_indices:
    rep = reps[idx]
    ok = True
    for _ in range(trials_per_rep):
        x = sample_coset_rep_plus_2E8(s, rep)
        if not in_rep_plus_2E8(x, rep):
            ok = False
            break
    print(f"rep index {idx}: coset membership check -> {ok}")


rep index 0: coset membership check -> True
rep index 1: coset membership check -> True
rep index 7: coset membership check -> True
rep index 42: coset membership check -> True
rep index 255: coset membership check -> True


In [7]:
# Test 2: bootstrap over all cosets
s = 2.2
masses, probs = prepare_bootstrap_distribution(s, reps)
print("sum of coset probabilities:", sum(probs))

N = 1000
counts = Counter()
violations = 0

for _ in range(N):
    x, idx = sample_E8_via_bootstrap(s, reps, masses)
    counts[idx] += 1
    if not in_E8(vector(ZZ, [ZZ(xi) for xi in x])):
        violations += 1

print("bootstrap samples not in E8:", violations)

print("top 8 cosets: theory vs empirical")
for i in sorted(range(len(reps)), key=lambda j: probs[j], reverse=True)[:8]:
    print(i, float(probs[i]), counts[i] / N)


sum of coset probabilities: 1.00000000000001
bootstrap samples not in E8: 0
top 8 cosets: theory vs empirical
0 0.02937109528974743 29/1000
22 0.005040178509550292 7/1000
19 0.005040178509550291 3/500
7 0.00504017850955029 1/125
16 0.00504017850955029 1/200
21 0.00504017850955029 1/250
1 0.005040178509550289 1/250
2 0.005040178509550289 1/250
